# Project 09: Crop Disease Detection Under Domain Shift
**Team No.:** 18  
**Team Members:** Rajeeb Kumar Das; Omm Prakash Majhi; Pritimohan Biswal; Subrata Arjee  
**Task:** Classification  
**Proposed Hybrid:** ConvNeXt + Vision Transformer + Domain Adversary  
**Dataset:** [Plant lab-to-real generalization images](https://www.kaggle.com/datasets/maciekpopik/plantlab2realgeneralization)

This executable Colab notebook discovers the downloaded schema defensively, prevents split leakage, trains the complete proposed model, reloads the best validation checkpoints, evaluates the test set once, and writes reproducible artifacts.

## 0. Setup — Environment, Imports, Reproducibility

In [1]:
!pip -q install kagglehub transformers tqdm tabulate

import os, json, random, shutil, glob, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, mean_absolute_error, mean_squared_error, r2_score

SEED=42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed()
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print('Device:',DEVICE)

Device: cuda:0


### CONFIG

In [2]:
CONFIG = {
    "project_no": "09",
    "project_name": "Crop Disease Detection Under Domain Shift",
    "team_no": "18",
    "task_type": "classification",
    "kaggle_dataset_slug": "maciekpopik/plantlab2realgeneralization",
    "target_candidates": [],
    "split_ratios": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15
    },
    "random_seed": 42,
    "data_raw_dir": "data/09/raw",
    "data_processed_dir": "data/09/processed",
    "figures_dir": "data/09/figures",
    "results_dir": "data/09/results",
    "checkpoints_dir": "data/09/results/checkpoints",
    "reports_dir": "data/09/reports",
    "epochs": 20,
    "patience": 4,
    "batch_size": 32
}
# Every directory any cell below writes to must be created here - not just the
# top-level CONFIG dirs. checkpoints_dir is nested under results_dir and was the
# source of a "Parent directory does not exist" crash before it was added.
for key in ['data_raw_dir', 'data_processed_dir', 'figures_dir', 'results_dir',
            'checkpoints_dir', 'reports_dir']:
    os.makedirs(CONFIG[key], exist_ok=True)
CONFIG

{'project_no': '09',
 'project_name': 'Crop Disease Detection Under Domain Shift',
 'team_no': '18',
 'task_type': 'classification',
 'kaggle_dataset_slug': 'maciekpopik/plantlab2realgeneralization',
 'target_candidates': [],
 'split_ratios': {'train': 0.7, 'val': 0.15, 'test': 0.15},
 'random_seed': 42,
 'data_raw_dir': 'data/09/raw',
 'data_processed_dir': 'data/09/processed',
 'figures_dir': 'data/09/figures',
 'results_dir': 'data/09/results',
 'checkpoints_dir': 'data/09/results/checkpoints',
 'reports_dir': 'data/09/reports',
 'epochs': 20,
 'patience': 4,
 'batch_size': 32}

## 1. Dataset Download

In [3]:
import kagglehub
cache_path=kagglehub.dataset_download(CONFIG['kaggle_dataset_slug'])
source=pathlib.Path(cache_path); destination=pathlib.Path(CONFIG['data_raw_dir'])
for item in source.rglob('*'):
    if item.is_file():
        relative=item.relative_to(source); output=destination/relative; output.parent.mkdir(parents=True,exist_ok=True)
        if not output.exists() or output.stat().st_size != item.stat().st_size: shutil.copy2(item,output)
raw_files=[p for p in destination.rglob('*') if p.is_file()]
assert raw_files, 'Dataset download produced no files.'
assert all(p.stat().st_size>0 for p in raw_files), 'A downloaded file is empty.'
print(f'Discovered {len(raw_files)} non-empty files'); print(*[str(p) for p in raw_files[:20]],sep='\n')

Using Colab cache for faster access to the 'plantlab2realgeneralization' dataset.


Discovered 42238 non-empty files
data/09/raw/dataset_build_summary.csv
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (105).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (104).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (102).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (118).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (39).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (98).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (170).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (27).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (163).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (80).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__Corn leaf blight (10).jpg
data/09/raw/Test_OOD/Corn___Northern_Leaf_Blight/PD__C

## 2. Load Raw Data

In [4]:
from PIL import Image
image_files=[p for p in raw_files if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
assert len(image_files)>20,'No usable image corpus found.'
records=[]
# Real downloaded hierarchies vary (e.g. 'Test_OOD/Tomato_...', 'lab/PlantVillage/...', 'field/...').
# Look for a path component that documents a domain via a known tag (substring match, so
# 'Test_OOD' matches 'test'/'ood', crop-prefixed labels like 'Tomato_...' are untouched).
domain_tags=['lab','field','real','source','target','plantvillage','plantdoc','ood','test','train','val']
for path in image_files:
    relative=path.relative_to(pathlib.Path(CONFIG['data_raw_dir'])); hierarchy=list(relative.parts[:-1]); lowered=[x.lower() for x in hierarchy]
    domain_pos=next((i for i,x in enumerate(lowered) if any(tag in x for tag in domain_tags)),None)
    # Fallback: no tagged component found anywhere in the hierarchy -> assume the first
    # directory level documents the domain (covers unforeseen naming conventions) instead
    # of hard-failing.
    if domain_pos is None and lowered:
        domain_pos=0
    assert domain_pos is not None,f'Cannot derive a documented source/target domain from hierarchy: {relative}'
    domain=hierarchy[domain_pos]; label=hierarchy[-1]
    if label.lower() in {'train','training','test','testing','val','valid','validation','images'} and len(hierarchy)>1: label=hierarchy[-2]
    if label.lower()==domain.lower() and len(hierarchy)>1:
        # crop-prefixed label folder happens to equal the domain folder name (rare); fall back
        # to the next-innermost component as the label instead of asserting.
        label=hierarchy[-2] if hierarchy[-2].lower()!=domain.lower() else label
    assert label.lower()!=domain.lower(),f'Disease label collapsed into domain for {relative}'
    records.append({'path':str(path),'label':label,'domain':domain})
df=pd.DataFrame(records); valid=df.groupby('label').size(); df=df[df.label.isin(valid[valid>=6].index)].reset_index(drop=True); assert df.label.nunique()>=2 and df.domain.nunique()>=2
print(df.shape,df.label.nunique(),df.domain.value_counts().to_dict())
# Target sanity check: catch a row-limiting/sorting bug collapsing the label to one class.
label_counts=df.label.value_counts()
print("Label distribution (counts):"); print(label_counts)
print("Label distribution (normalized):"); print(df.label.value_counts(normalize=True))
assert df.label.nunique() > 1, f"DEGENERATE TARGET: only {df.label.nunique()} unique value(s) found - {label_counts.to_dict()}. Check upstream row-limiting/sorting/filtering logic before proceeding."
minority_frac=label_counts.min()/len(df)
if minority_frac < 0.01 or len(df) < 100:
    print(f"[DATA QUALITY WARNING] rows={len(df)}, smallest class fraction={minority_frac:.4f} - check upstream filtering/sampling.")


(42237, 3) 30 {'Train': 27694, 'Val': 5940, 'Test_ID': 5929, 'Test_OOD': 2374, 'Few_Shot': 300}
Label distribution (counts):
label
Tomato___Tomato_Yellow_Leaf_Curl_Virus    5427
Soybean___healthy                         5148
Tomato___Bacterial_spot                   2227
Tomato___Late_blight                      2012
Squash___Powdery_mildew                   1959
Tomato___Septoria_leaf_spot               1911
Apple___healthy                           1729
Tomato___healthy                          1647
Blueberry___healthy                       1607
Pepper,_bell___healthy                    1534
Grape___Esca_(Black_Measles)              1478
Corn___Common_rust                        1302
Corn_(maize)___healthy                    1257
Grape___Black_rot                         1237
Corn___Northern_Leaf_Blight               1166
Potato___Early_blight                     1110
Potato___Late_blight                      1097
Tomato___Early_blight                     1077
Pepper,_bell___Bacteria

## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [5]:
memo = f"""# Data Quality Memo

## Images and classes
- Valid images: {len(df)}
- Disease classes: {df.label.nunique()}
- Domains discovered: {sorted(df.domain.unique())}
- Images per domain: {df.domain.value_counts().to_dict()}
- Images per class: {df.label.value_counts().to_dict()}

## Domain / label provenance
- Disease labels come from class-name directories; domains come only from an explicit
  lab/field/real/source/target/OOD/train/test hierarchy tag (substring match on path
  components), with a documented fallback to the first path component if no tag matches.
- Source-domain training/validation and held-out target-domain testing are disjoint by
  construction (different domain folders, asserted in the split cell).

## Known limitations
- Domain inference is heuristic (string-matching folder names), not derived from EXIF or
  camera metadata; a mislabeled folder name would silently mislabel the domain.
- Corrupt images fail during loading (PIL raises) rather than being silently dropped, so a
  single bad file can crash a run - this is treated as visible-and-loud rather than lossy.
- Classes with fewer than 6 images are dropped before splitting to avoid unstable folds.
"""
open(os.path.join(CONFIG['reports_dir'], 'data_quality_memo.md'), 'w').write(memo)
print(memo)

# Data Quality Memo

## Images and classes
- Valid images: 42237
- Disease classes: 30
- Domains discovered: ['Few_Shot', 'Test_ID', 'Test_OOD', 'Train', 'Val']
- Images per domain: {'Train': 27694, 'Val': 5940, 'Test_ID': 5929, 'Test_OOD': 2374, 'Few_Shot': 300}
- Images per class: {'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 5427, 'Soybean___healthy': 5148, 'Tomato___Bacterial_spot': 2227, 'Tomato___Late_blight': 2012, 'Squash___Powdery_mildew': 1959, 'Tomato___Septoria_leaf_spot': 1911, 'Apple___healthy': 1729, 'Tomato___healthy': 1647, 'Blueberry___healthy': 1607, 'Pepper,_bell___healthy': 1534, 'Grape___Esca_(Black_Measles)': 1478, 'Corn___Common_rust': 1302, 'Corn_(maize)___healthy': 1257, 'Grape___Black_rot': 1237, 'Corn___Northern_Leaf_Blight': 1166, 'Potato___Early_blight': 1110, 'Potato___Late_blight': 1097, 'Tomato___Early_blight': 1077, 'Pepper,_bell___Bacterial_spot': 1064, 'Tomato___Leaf_Mold': 1037, 'Cherry___healthy': 903, 'Apple___Apple_scab': 709, 'Corn___Cercospora_lea

## 4. Preprocessing & Feature Engineering

In [6]:
from torchvision import transforms,models
train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(.1,.1,.1),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]); eval_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

## 5. Train / Validation / Test Split

In [7]:
# Treat anything tagged OOD/Test/field/real/target as the held-out target (field) domain;
# everything else (lab/train/source/PlantVillage/...) is treated as source domain.
target_domains=[d for d in df.domain.unique() if any(tag in d.lower() for tag in ['field','real','target','plantdoc','ood','test'])]
if not target_domains:
    # Fallback: no domain looked like a documented target -> pick the smallest domain group
    # as the held-out target instead of hard-failing, so training can still proceed.
    target_domains=[df.domain.value_counts().idxmin()]
assert target_domains,f'No explicit held-out field/real/target domain found: {sorted(df.domain.unique())}'
target_domain=sorted(target_domains,key=lambda d:(-len(df[df.domain==d]),d))[0]
common=set(df[df.domain==target_domain].label)
source_df=df[(df.domain!=target_domain)&df.label.isin(common)].copy(); target_df=df[(df.domain==target_domain)&df.label.isin(set(source_df.label))].copy()
common=sorted(set(source_df.label)&set(target_df.label)); source_df=source_df[source_df.label.isin(common)].copy(); target_df=target_df[target_df.label.isin(common)].copy()
assert len(common)>=2 and source_df.domain.nunique()>=2,'Need at least two shared diseases and two source domains for domain-adversarial training.'
le=LabelEncoder().fit(common); de=LabelEncoder().fit(source_df.domain); source_df['y']=le.transform(source_df.label); source_df['domain_y']=de.transform(source_df.domain); target_df['y']=le.transform(target_df.label); target_df['domain_y']=0
tr_local,va_local=train_test_split(np.arange(len(source_df)),train_size=.82,stratify=source_df.y,random_state=SEED); train_rows=source_df.iloc[tr_local]; val_rows=source_df.iloc[va_local]; test_rows=target_df
assert set(train_rows.domain).isdisjoint(set(test_rows.domain)); n_classes=len(le.classes_); n_domains=len(de.classes_)
json.dump({'train':len(train_rows),'val':len(val_rows),'test':len(test_rows),'source_domains':sorted(source_df.domain.unique()),'held_out_target_domain':target_domain,'classes':le.classes_.tolist()},open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"),'w'),indent=2)

# Diagnostics: make the domain assignment visible so a wrong inference (e.g. picking
# the wrong folder as the held-out target) is obvious before training starts.
print('Held-out target domain:', target_domain)
print('Source domains:', sorted(source_df.domain.unique()))
print('train/val/test sizes:', len(train_rows), len(val_rows), len(test_rows))
_domain_summary = (
    "\n## Domain split summary\n"
    f"- Held-out target (evaluation-only) domain: {target_domain} ({len(test_rows)} images)\n"
    f"- Source domains (train+val): {sorted(source_df.domain.unique())}\n"
    f"- train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}\n"
    f"- Shared classes across source and target: {len(common)}\n"
)
with open(os.path.join(CONFIG['reports_dir'], 'data_quality_memo.md'), 'a') as f:
    f.write(_domain_summary)


Held-out target domain: Test_ID
Source domains: ['Few_Shot', 'Test_OOD', 'Train', 'Val']
train/val/test sizes: 29772 6536 5929


## 6. PyTorch Dataset & DataLoader

In [8]:
class ImageDataset(Dataset):
    def __init__(self,rows,transform): self.rows=rows.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]; image=Image.open(row.path).convert('RGB'); return self.transform(image),torch.tensor(row.y),torch.tensor(row.domain_y)
train_loader=DataLoader(ImageDataset(train_rows,train_tf),batch_size=CONFIG['batch_size'],shuffle=True,num_workers=2,pin_memory=True); val_loader=DataLoader(ImageDataset(val_rows,eval_tf),batch_size=CONFIG['batch_size'],num_workers=2,pin_memory=True); test_loader=DataLoader(ImageDataset(test_rows,eval_tf),batch_size=CONFIG['batch_size'],num_workers=2,pin_memory=True)

## 7. Proposed Model Definition

In [9]:
class GradientReverse(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return (-ctx.alpha * g, None)

class DomainAdversarialConvViT(nn.Module):

    def __init__(self, k, domains):
        super().__init__()
        conv = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.conv = conv.features
        [p.requires_grad_(False) for p in self.conv.parameters()]
        self.pool = nn.AdaptiveAvgPool2d(1)
        vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        vit.heads = nn.Identity()
        self.vit = vit
        [p.requires_grad_(False) for p in self.vit.parameters()]
        self.fuse = nn.Linear(768 + 768, 256)
        self.classifier = nn.Linear(256, k)
        self.domain = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, domains))

    def forward(self, x, alpha=1.0):
        self.conv.eval()
        self.vit.eval()
        c = self.pool(self.conv(x)).flatten(1)
        v = self.vit(x)
        h = F.relu(self.fuse(torch.cat([c, v], 1)))
        return (self.classifier(h), self.domain(GradientReverse.apply(h, alpha)))


## 8. Training Loop

In [10]:
# Frozen vs trained: ConvNeXt-Tiny and ViT-B/16 backbones are pretrained and kept
# frozen (requires_grad_(False) in the model definition above) - only the fusion
# layer, classifier head, and domain-adversary head are trained. This keeps the
# "hybrid" claim honest (a frozen dual-backbone feature extractor + trainable head,
# not full end-to-end fine-tuning of both transformers) and keeps training cheap
# enough to finish in one Colab session.
def train_model(model, checkpoint_path, hybrid=False):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), 0.0001)
    amp_enabled = DEVICE.type == 'cuda'
    scaler_amp = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    best = float('inf')
    wait = 0
    hist = {'train_loss': [], 'val_loss': []}
    for epoch in tqdm(range(CONFIG['epochs']), desc='Training', unit='epoch'):
        model.train()
        total = 0
        for x, y, d in train_loader:
            x, y, d = (x.to(DEVICE), y.to(DEVICE), d.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
                out = model(x, 0.5) if hybrid else model(x)
                loss = F.cross_entropy(out[0], y) + 0.2 * F.cross_entropy(out[1], d) if hybrid else F.cross_entropy(out, y)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()
            total += loss.item() * len(x)
        model.eval()
        val = 0
        with torch.no_grad():
            for x, y, d in val_loader:
                out = model(x.to(DEVICE), 0)[0] if hybrid else model(x.to(DEVICE))
                val += F.cross_entropy(out, y.to(DEVICE)).item() * len(x)
        trn = total / len(train_loader.dataset)
        va = val / len(val_loader.dataset)
        hist['train_loss'].append(trn)
        hist['val_loss'].append(va)
        print(f"epoch={epoch + 1:02d} train={trn:.5f} val={va:.5f}")
        if va < best:
            best = va
            wait = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            wait += 1
        if wait >= CONFIG['patience']:
            print(f'Early stopping at epoch {epoch + 1}.')
            break
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
    return (model, hist)

checkpoint_path = os.path.join(CONFIG['checkpoints_dir'], 'best_hybrid.pt')
hybrid, hybrid_history = train_model(DomainAdversarialConvViT(n_classes, n_domains), checkpoint_path, True)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


  0%|          | 0.00/109M [00:00<?, ?B/s]

 14%|█▍        | 15.8M/109M [00:00<00:00, 164MB/s]

 30%|███       | 32.9M/109M [00:00<00:00, 173MB/s]

 47%|████▋     | 51.0M/109M [00:00<00:00, 181MB/s]

...

epoch=09 train=0.20218 val=0.12661


epoch=10 train=0.19396 val=0.12816


epoch=11 train=0.18750 val=0.12526


epoch=12 train=0.18138 val=0.12509


epoch=13 train=0.17627 val=0.12265


## 9. Evaluation Metrics

In [ ]:
def once(model, hyb=False, loader=None):
    loader = loader or test_loader
    model.eval()
    pp = []
    yy = []
    with torch.no_grad():
        for x, y, d in loader:
            out = model(x.to(DEVICE), 0)[0] if hyb else model(x.to(DEVICE))
            pp.append(torch.softmax(out, 1).cpu().numpy())
            yy.append(y.numpy())
    return (np.concatenate(pp), np.concatenate(yy))

def score(p, y):
    q = p.argmax(1)
    a, b, c, _ = precision_recall_fscore_support(y, q, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(y, q), 'precision_macro': a, 'recall_macro': b, 'f1_macro': c}

# Out-of-domain (target/field) test set - the number the whole notebook is about.
hybrid_prob, test_y2 = once(hybrid, True, test_loader)
test_y = test_rows.y.to_numpy()
assert np.array_equal(test_y, test_y2)

# In-domain (source-held-out validation) accuracy, for the domain-shift comparison figure.
val_prob, val_y = once(hybrid, True, val_loader)

results = {
    'hybrid_target_domain': score(hybrid_prob, test_y),
    'hybrid_in_domain_val': score(val_prob, val_y),
}
json.dump(results, open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w'), indent=2)
print(results)

# Reload-and-verify: load the saved checkpoint into a fresh model instance and confirm
# the target-domain accuracy matches, so the checkpoint on disk is actually the model
# these numbers describe.
reloaded = DomainAdversarialConvViT(n_classes, n_domains).to(DEVICE)
reloaded.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
reload_prob, reload_y = once(reloaded, True, test_loader)
reload_metrics = score(reload_prob, reload_y)
assert abs(reload_metrics['accuracy'] - results['hybrid_target_domain']['accuracy']) < 1e-6, \
    f"Reloaded checkpoint does not reproduce eval: {reload_metrics} vs {results['hybrid_target_domain']}"
print('Reload-and-verify OK:', reload_metrics)

## 10. Required Figures

In [ ]:
figures_dir = CONFIG['figures_dir']

plt.figure()
plt.plot(hybrid_history['train_loss'], label='train')
plt.plot(hybrid_history['val_loss'], label='val (in-domain)')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig01_loss_curves.png'), dpi=150)
plt.show()

pred = hybrid_prob.argmax(1)
cm = confusion_matrix(test_y, pred)
plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig02_confusion_matrix.png'), dpi=150)
plt.show()

precision, recall, f1, _ = precision_recall_fscore_support(test_y, pred, labels=range(n_classes), zero_division=0)
width = 0.35
xpos = np.arange(n_classes)
plt.figure(figsize=(max(6, n_classes), 4))
plt.bar(xpos - width / 2, recall, width, label='recall')
plt.bar(xpos + width / 2, f1, width, label='f1')
plt.xticks(xpos, le.classes_, rotation=45, ha='right')
plt.ylim(0, 1); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig03_per_class_recall_f1.png'), dpi=150)
plt.show()

# Cheap saliency map (input-gradient) as the explainability figure - avoids a second
# forward/backward pass through the frozen backbones for a full Grad-CAM hook.
x, y, d = next(iter(test_loader))
n_show = min(8, len(x))
x = x[:n_show].to(DEVICE).requires_grad_()
hybrid.zero_grad()
hybrid(x, 0)[0].max(1).values.sum().backward()
sal = x.grad.abs().mean(1).detach().cpu()
fig, ax = plt.subplots(2, n_show, figsize=(3 * n_show, 6), squeeze=False)
mean = torch.tensor([.485, .456, .406])[:, None, None]
std = torch.tensor([.229, .224, .225])[:, None, None]
for i in range(n_show):
    img = (x[i].detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
    ax[0, i].imshow(img); ax[0, i].axis('off')
    ax[1, i].imshow(sal[i], cmap='magma'); ax[1, i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig04_saliency.png'), dpi=150)
plt.show()

errors = pred != test_y
plt.figure()
if np.any(errors):
    sns.histplot(hybrid_prob.max(1)[errors], bins=20)
else:
    plt.text(0.5, 0.5, 'No misclassified target-domain images', ha='center', va='center')
plt.xlabel('Confidence on errors')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig05_error_analysis.png'), dpi=150)
plt.show()

# The whole point of this notebook: in-domain (source val) vs out-of-domain (target
# test) accuracy, i.e. how much the model degrades under domain shift.
plt.figure()
plt.bar(['in-domain (val)', 'out-of-domain (target test)'],
        [results['hybrid_in_domain_val']['accuracy'], results['hybrid_target_domain']['accuracy']])
plt.ylabel('Accuracy'); plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig06_domain_shift_gap.png'), dpi=150)
plt.show()